# Lesson 9: Multi-Agent Systems

## Lab 09: Build a Research Team Agent

In Lesson 8 you built one agent that chose between different paths. In this activity you build a **team**: several specialised agents that work together on a single research question, coordinated by a supervisor.

| Team member | Job |
|---|---|
| **Researcher** | Gathers facts with tools: the course notes and Wikipedia |
| **Analyst** | Organises the facts into key points, gaps and an outline |
| **Writer** | Writes a short report from the analysis |
| **Reviewer** | Checks the report and approves it, or sends it back with feedback |
| **Supervisor** | Decides who works next, and when the team is finished |

The agents never talk to each other directly. They read from and write to a **shared state**, like a team working on the same whiteboard.

## Learning objectives

By the end of this activity, you should be able to:

1. Explain when multiple agents are better than one, and when they aren't.
2. Design specialised agents and delegate work between them.
3. Explain the supervisor architecture.
4. Use shared state for agent communication.
5. Build and test a supervisor-led multi-agent workflow with LangGraph.

**Time:** about 2 hours
**You'll need:** a Gemini API key, and the Lesson 8 LangGraph activity completed.

> **How to use this notebook:** run the cells from top to bottom. Read each explanation before running the code below it. Cells marked **Try it** invite you to change something and run again.

## 1. Why use multiple agents?

Imagine asking **one** agent to research a topic, organise the findings, write a report, and then check its own work. It has a very long prompt, lots of tools, and four different jobs. That causes problems:

- **Confusion:** the more instructions and tools an agent has, the more likely it is to mix them up.
- **No fresh eyes:** an agent checking its own work tends to approve it.
- **Hard to improve:** if the reports are badly written, which part of the giant prompt do you fix?

A multi-agent system splits the work, the way a newsroom has reporters, editors and fact-checkers:

| One agent does everything | A team of specialists |
|---|---|
| One long prompt with every instruction | A short, focused prompt per agent |
| Every tool available all the time | Each agent gets only the tools it needs |
| Checks its own work | A separate Reviewer gives an independent check |
| Hard to tell which part failed | You can test and improve each agent on its own |

### But don't overdo it

More agents means more LLM calls, more cost, more time, and more to debug. Remember this from the Lesson 5 course notes: *start with a single agent, and add more only when you need them.* A research report with clearly separate stages is a good fit. A simple question-and-answer bot is not.

## 2. Specialised agents and delegation

A **specialised agent** has one clear job, defined by three things:

| Part | Example: the Researcher |
|---|---|
| **Instructions** | "Gather 4–8 facts with sources. Don't write the report." |
| **Tools** | `search_course_notes`, `search_wikipedia` |
| **Output** | Bullet-point notes, each with a source |

Specialists can be very different kinds of agent. Our Researcher is a full tool-using agent with its own loop. The Analyst and Reviewer return **structured output** (Lessons 2 and 8). The Writer is a simple prompt-plus-model chain.

**Delegation** means giving a piece of work to the agent best suited for it, instead of doing it yourself. In our team, the supervisor delegates every task. It never researches or writes anything itself.

## 3. The supervisor architecture

There are three common ways to organise a team of agents:

| Architecture | How it works | Good for |
|---|---|---|
| **Sequential pipeline** | A → B → C → D, always in the same order | Fixed processes that never need to go back |
| **Supervisor** | A central coordinator decides who works next; everyone reports back to it | Processes that sometimes need to repeat a step |
| **Network** | Agents hand work directly to each other | Open-ended collaboration (harder to control and debug) |

We use a **supervisor**, because our team sometimes needs to go back: when the Reviewer rejects a report, the Writer must revise it.

```text
                         START
                           ↓
            ┌──────────► SUPERVISOR ◄──────────┐
            │      ┌────────┼────────┬─────────┤
            │      ↓        ↓        ↓         ↓
            │  Researcher Analyst  Writer  Reviewer
            │      │        │        │         │
            └──────┴────────┴────────┴─────────┘
                    (everyone reports back)
                           ↓
                        finish → END
```

This is the **hub-and-spoke** pattern: the supervisor is the hub, and the agents are the spokes.

A supervisor can be **code** (clear rules) or an **LLM** (it reads the situation and decides). We start with rules, because they're predictable and easy to test. In the challenge, you'll try an LLM supervisor.

## 4. Agent communication

Agents need to pass work to each other: the Analyst needs the Researcher's notes, the Writer needs the Analyst's outline, and so on. There are two main ways:

| Approach | How it works |
|---|---|
| **Direct messages** | Agents send messages to each other, like a group chat |
| **Shared state** (a "blackboard") | Every agent reads from and writes to one shared record, like a whiteboard in a meeting room |

We use **shared state**. Each agent:

1. **reads** the parts it needs (the Writer reads `notes` and `analysis`),
2. **does its job**,
3. **writes** only its own part back (the Writer writes `draft`).

Nobody overwrites anyone else's work, and the supervisor can see the whole picture at any time.

We also keep a **team log**: each agent adds one line saying what it did. It's like a message board that shows how the team communicated, and it's very useful for debugging.

### Setup

Install the packages. These are the same as Lesson 8.

In [ ]:
!pip -q install langchain langgraph langchain-google-genai

In [ ]:
import os
import getpass

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")

print("API key loaded.")

## 5. Shared state

The shared state is a `TypedDict`, just like `HelpdeskState` in Lesson 8. The comments show **which team member writes each field**.

`team_log` uses a new reducer, `operator.add`. For lists, `+` joins them together, so each agent's new log line is **added** to the list instead of replacing it. It's the same idea as `add_messages` in Lesson 8, but for a plain list of strings.

In [ ]:
import operator
from typing import Annotated, TypedDict


class ResearchState(TypedDict):
    topic: str                                     # the research question
    notes: str                                     # Researcher -> facts with sources
    analysis: dict                                 # Analyst    -> key_points, gaps, outline
    draft: str                                     # Writer     -> the current report
    revision: int                                  # how many drafts the Writer has written
    review: dict                                   # Reviewer   -> approved, score, feedback, draft_number
    next_agent: str                                # Supervisor -> who works next
    final_report: str                              # the approved (or last) report
    team_log: Annotated[list[str], operator.add]   # every agent adds a line: the team's message board


# What the reducer does: old list + new list.
print(operator.add(["Researcher: gathered notes."], ["Analyst: found 4 key points."]))

Every other field has no reducer, so a new value simply **replaces** the old one. That's what we want for `draft`: each revision replaces the previous draft.

## 6. Design the research team

Before writing code, design the team on paper. For each agent, decide what it **reads** from the shared state and what it **writes**:

| Agent | Reads | Writes | How it works |
|---|---|---|---|
| **Researcher** | `topic` | `notes` | Tool-using agent (`create_agent`) with two search tools |
| **Analyst** | `topic`, `notes` | `analysis` | Structured output: key points, gaps, outline |
| **Writer** | `topic`, `notes`, `analysis`, `review` | `draft`, `revision` | Prompt + model chain; uses the feedback when revising |
| **Reviewer** | `topic`, `notes`, `draft` | `review` | Structured output: approved, score, feedback |
| **Supervisor** | everything | `next_agent` | Rules in plain Python |
| **finish** | `draft`, `review` | `final_report` | Publishes the last draft |

Every agent also adds one line to `team_log`.

Two design decisions keep the team safe:

- **A revision limit.** The Reviewer could keep rejecting forever, so the Writer may revise at most `MAX_REVISIONS` times.
- **Grounded facts.** Only the Researcher uses tools. Everyone else must use only the Researcher's notes. That stops facts being invented later in the process.

Now let's set up the team's equipment: the model, the course notes and the Researcher's tools.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

MODEL = "gemini-3.8-flash"
llm = ChatGoogleGenerativeAI(model=MODEL)

MAX_REVISIONS = 2   # the Writer may revise at most this many times after the first draft
MAX_WORDS = 250     # target length for the report

print("Model ready:", MODEL)

The **course notes** are the same 8 documents from Lessons 5 and 8, loaded into a LangChain retriever.

In [ ]:
KNOWLEDGE_BASE = [
    {
        "id": "kb-01",
        "title": "What is an AI agent?",
        "text": (
            "An AI agent is a system that uses a language model to pursue a goal by deciding what to do next. "
            "Unlike a chatbot, which only replies to messages, an agent can take actions with tools, look at the "
            "results, and choose its next step. A workflow follows a fixed path written by a developer; an agent "
            "chooses its own path. Most real agents are assisted: a human approves important actions such as "
            "payments or sending emails."
        ),
    },
    {
        "id": "kb-02",
        "title": "Tools and function calling",
        "text": (
            "A tool is a function an agent can ask to use, such as a calculator, a web search or an API. The model "
            "never runs the tool itself. It returns a function call containing the tool name and arguments; your "
            "Python code runs the function and sends the result back. Each tool is described with a schema: a name, "
            "a description and parameters. The model chooses tools using only these descriptions, so clear "
            "descriptions matter more than clever code."
        ),
    },
    {
        "id": "kb-03",
        "title": "The ReAct loop and stopping rules",
        "text": (
            "ReAct stands for Reason and Act. The agent reasons about what it needs, acts by calling a tool, "
            "observes the result, and repeats until it can answer. Every agent loop needs three stopping rules: "
            "stop when the model makes no more tool calls, stop when the step budget runs out, and block repeated "
            "identical calls with a loop guard. Without them, an agent can loop forever and waste money."
        ),
    },
    {
        "id": "kb-04",
        "title": "Agent memory",
        "text": (
            "Short-term memory is the conversation history the agent sends to the model on every turn. It is limited "
            "by the context window, the maximum number of tokens the model can read at once. Long-term memory lives "
            "outside the model, for example in a database, and is retrieved only when needed. Retrieval lets an "
            "agent use far more information than fits in its context window."
        ),
    },
    {
        "id": "kb-05",
        "title": "Retrieval-Augmented Generation (RAG)",
        "text": (
            "RAG has three steps: retrieve relevant documents, augment the prompt with them, and generate an answer "
            "grounded in that text. It reduces hallucination because the model answers from real sources it can "
            "cite. Unlike fine-tuning, RAG does not change the model's weights; to update its knowledge you simply "
            "update the documents. In agentic RAG, retrieval is a tool, and the agent decides when to search and "
            "what to search for."
        ),
    },
    {
        "id": "kb-06",
        "title": "Embeddings and vector databases",
        "text": (
            "An embedding is a list of numbers that represents the meaning of a piece of text. Texts with similar "
            "meanings have embeddings that are close together, even if they share no words, so 'car' and 'vehicle' "
            "end up near each other. Closeness is usually measured with cosine similarity. A vector database such "
            "as ChromaDB stores embeddings and quickly finds the ones closest to a query. This is called semantic search."
        ),
    },
    {
        "id": "kb-07",
        "title": "Multi-agent systems",
        "text": (
            "A multi-agent system splits work between several agents with different roles, such as a planner, a "
            "researcher and a writer. An orchestrator agent assigns tasks and combines the results. Agents pass work "
            "to each other through handoffs. Multiple agents help with large or varied tasks, but they cost more and "
            "are harder to debug, so start with a single agent and add more only when you need them."
        ),
    },
    {
        "id": "kb-08",
        "title": "Agent safety and guardrails",
        "text": (
            "Agents act in the real world, so they need guardrails. Give each agent only the tools it needs, a "
            "principle called least privilege. Ask a human to approve risky actions such as payments, deletions or "
            "sending messages. Treat text that comes back from tools and documents as data, never as instructions; "
            "attackers can hide commands in web pages, an attack called prompt injection. Log every step so you can "
            "audit what the agent did."
        ),
    },
]


from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

documents = [Document(page_content=d["text"], metadata={"title": d["title"]}) for d in KNOWLEDGE_BASE]
vector_store = InMemoryVectorStore.from_documents(documents, GoogleGenerativeAIEmbeddings(model="gemini-embedding-001"))
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

print("Course notes ready:", len(documents), "documents.")

The Researcher gets two tools. Each one labels its results with the source, so the Researcher can cite them. As always, tools return an error message instead of crashing.

In [ ]:
import requests
from langchain_core.tools import tool


@tool
def search_course_notes(query: str) -> str:
    """Search the course notes about agentic AI: agents, tools, the ReAct loop, memory, RAG,
    embeddings, multi-agent systems and agent safety. Returns the most relevant notes with their titles."""
    try:
        docs = retriever.invoke(query)
        return "\n\n".join(f"[Course notes: {d.metadata['title']}]\n{d.page_content}" for d in docs)
    except Exception as e:
        return f"Error: course notes search failed: {e}"


@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia for background information on any topic. Returns a short summary and the article title."""
    headers = {"User-Agent": "AgenticAICourse/1.0 (educational lab)"}
    try:
        search = requests.get("https://en.wikipedia.org/w/api.php", headers=headers, timeout=10, params={
            "action": "query", "list": "search", "srsearch": query, "format": "json", "srlimit": 1}).json()
        hits = search.get("query", {}).get("search", [])
        if not hits:
            return f"No Wikipedia article found for '{query}'."
        title = hits[0]["title"]
        page = requests.get("https://en.wikipedia.org/api/rest_v1/page/summary/"
                            + requests.utils.quote(title.replace(" ", "_")), headers=headers, timeout=10).json()
        return f"[Wikipedia: {title}]\n{page.get('extract', '')[:1200]}"
    except Exception as e:
        return f"Error: Wikipedia search failed: {e}"


RESEARCH_TOOLS = [search_course_notes, search_wikipedia]


print(search_course_notes.invoke({"query": "agent memory"})[:300], "...\n")
print(search_wikipedia.invoke({"query": "multi-agent system"})[:300], "...")

## 7. Build the Researcher agent

The Researcher is a complete **tool-using agent**. We build it with LangChain's `create_agent` (Lesson 8), which runs the Reason → Act → Observe loop for us.

Its instructions say what to do **and what not to do**: the Researcher gathers facts and leaves the writing to the Writer. Clear boundaries keep specialists from stepping on each other's work.

The node function follows the pattern from Lesson 8: read from the state, do the job, and return only what changed.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage


RESEARCHER_PROMPT = """You are the Researcher on a research team. Gather facts for the research question.
- Search the course notes first, then Wikipedia for wider background. Make 2 to 4 searches in total.
- Reply with 4 to 8 bullet points. Each bullet is one fact, followed by its source in brackets,
  for example: (Source: Course notes: Agent memory) or (Source: Wikipedia: Retrieval-augmented generation).
- Only include facts you found with the tools. Don't write the report; that's the Writer's job."""

research_agent = create_agent(model=llm, tools=RESEARCH_TOOLS, system_prompt=RESEARCHER_PROMPT)


def researcher(state: ResearchState) -> dict:
    # Gather facts about the topic using tools.
    result = research_agent.invoke({"messages": [HumanMessage(content=f"Research question: {state['topic']}")]})
    notes = result["messages"][-1].text
    searches = sum(len(getattr(m, "tool_calls", None) or []) for m in result["messages"])
    print(f"  [Researcher] made {searches} searches")
    return {"notes": notes, "team_log": [f"Researcher: gathered notes using {searches} searches."]}


print("Researcher ready.")

Let's test the Researcher on its own, before it joins the team. We create a fresh task, run the node, and merge its update into our test state.

When the full graph runs, LangGraph does this merging for us, including adding to `team_log`.

In [ ]:
def new_task(topic: str) -> dict:
    return {"topic": topic, "notes": "", "analysis": {}, "draft": "", "revision": 0,
            "review": {}, "next_agent": "", "final_report": "", "team_log": []}


demo = new_task("How do AI agents use memory?")
demo.update(researcher(demo))

print("\nNOTES:\n")
print(demo["notes"])

Each fact should end with its source. Those labels let the Writer cite sources, and let the Reviewer check that nothing was invented.

## 8. Build the Analyst agent

The Analyst turns a list of facts into a plan. It uses **structured output** with three fields:

- `key_points`: the most important facts that answer the question,
- `gaps`: anything important the notes don't cover,
- `outline`: the section headings for the report.

Spotting gaps is a real analyst skill. It tells the team, and the reader, what the research **didn't** find.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate


class Analysis(BaseModel):
    """The Analyst's output."""
    key_points: list[str] = Field(description="The 3-5 most important points that answer the question")
    gaps: list[str] = Field(description="Anything important the notes don't cover (can be empty)")
    outline: list[str] = Field(description="3-4 section headings for the report, in order")


ANALYST_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You are the Analyst on a research team. Organise the Researcher's notes. "
               "Use only the notes; don't add new facts."),
    ("human", "Research question: {topic}\n\nResearcher's notes:\n{notes}"),
])


def analyst(state: ResearchState) -> dict:
    # Organise the notes into key points, gaps and an outline.
    chain = ANALYST_PROMPT | llm.with_structured_output(Analysis)
    analysis = chain.invoke({"topic": state["topic"], "notes": state["notes"]})
    print(f"  [Analyst] {len(analysis.key_points)} key points, outline: {analysis.outline}")
    return {"analysis": analysis.model_dump(),
            "team_log": [f"Analyst: found {len(analysis.key_points)} key points and {len(analysis.gaps)} gaps."]}


demo.update(analyst(demo))

print("\nKey points:", *demo["analysis"]["key_points"], sep="\n  - ")
print("\nGaps:", *(demo["analysis"]["gaps"] or ["(none)"]), sep="\n  - ")
print("\nOutline:", demo["analysis"]["outline"])

## 9. Build the Writer agent

The Writer turns the analysis into a short Markdown report. It reads the most fields of anyone: the topic, the notes, the outline, the key points and any feedback.

Notice the `feedback` slot. On the **first** draft there's no review yet, so it says "none". On a **revision**, it contains the Reviewer's feedback. That's how the Reviewer and Writer communicate through the shared state.

Each time it writes, the Writer adds 1 to `revision`, so everyone knows which draft is current.

In [ ]:
WRITER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You are the Writer on a research team. Write a short report for beginners, in Markdown, "
               "under {max_words} words. Follow the outline as section headings. Use only facts from the notes. "
               "End with a 'Sources' section listing the sources you used."),
    ("human", "Research question: {topic}\n\nOutline: {outline}\n\nKey points:\n{key_points}\n\n"
              "Researcher's notes:\n{notes}\n\nReviewer feedback to fix (if any): {feedback}"),
])


def writer(state: ResearchState) -> dict:
    # Write the report, or revise it using the Reviewer's feedback.
    feedback = state["review"].get("feedback", "") if state.get("review") else ""
    chain = WRITER_PROMPT | llm
    reply = chain.invoke({
        "topic": state["topic"],
        "max_words": MAX_WORDS,
        "outline": ", ".join(state["analysis"]["outline"]),
        "key_points": "\n".join(f"- {p}" for p in state["analysis"]["key_points"]),
        "notes": state["notes"],
        "feedback": feedback or "none",
    })
    revision = state.get("revision", 0) + 1
    action = "wrote the first draft" if revision == 1 else f"revised the report (draft {revision})"
    print(f"  [Writer] {action}, {len(reply.text.split())} words")
    return {"draft": reply.text, "revision": revision, "team_log": [f"Writer: {action}."]}


demo.update(writer(demo))

print("\nDRAFT:\n")
print(demo["draft"])

## 10. Build the Reviewer agent

The Reviewer gives an independent check against **four clear criteria**. Vague instructions like "check if it's good" make a reviewer that approves everything, so the criteria are spelled out.

Its structured output has `approved`, `score` and `feedback`. We also record **which draft** it reviewed (`draft_number`). The supervisor uses that to tell whether the latest draft has been reviewed yet.

In [ ]:
class Review(BaseModel):
    """The Reviewer's output."""
    approved: bool = Field(description="True only if the report meets every criterion")
    score: int = Field(description="Overall quality from 1 (poor) to 10 (excellent)")
    feedback: str = Field(description="Specific changes the Writer should make; empty if approved")


REVIEWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You are the Reviewer on a research team. Check the report against these criteria:\n"
               "1. It answers the research question.\n"
               "2. Every fact is supported by the Researcher's notes (no invented facts).\n"
               "3. It ends with a Sources section.\n"
               "4. It is under {max_words} words and easy for a beginner to understand.\n"
               "Approve only if all four are met. Otherwise give short, specific feedback."),
    ("human", "Research question: {topic}\n\nResearcher's notes:\n{notes}\n\nReport to review:\n{draft}"),
])


def reviewer(state: ResearchState) -> dict:
    # Check the report against the criteria and approve it or ask for changes.
    chain = REVIEWER_PROMPT | llm.with_structured_output(Review)
    review = chain.invoke({"topic": state["topic"], "notes": state["notes"],
                           "draft": state["draft"], "max_words": MAX_WORDS})
    verdict = "approved" if review.approved else "asked for changes"
    print(f"  [Reviewer] {verdict} (score {review.score}/10){'' if review.approved else ': ' + review.feedback}")
    return {"review": {**review.model_dump(), "draft_number": state["revision"]},
            "team_log": [f"Reviewer: {verdict} (draft {state['revision']}, score {review.score}/10)."]}


demo.update(reviewer(demo))
print("\nReview:", demo["review"])

> **Try it:** make the Reviewer stricter. For example, add a fifth criterion: "uses at least one source from Wikipedia". Re-run the cell. Does the verdict change?

## 11. Create a simple supervisor

The supervisor looks at the shared state and decides **who works next**. It's plain Python: a list of rules checked in order, like a project manager's checklist.

| Rule (checked in order) | Next |
|---|---|
| No notes yet | Researcher |
| No analysis yet | Analyst |
| No draft yet | Writer |
| The latest draft hasn't been reviewed | Reviewer |
| The review approved it | finish |
| Rejected, and revisions are still allowed | Writer (to revise) |
| Rejected, and the revision limit is reached | finish |

The last rule is a **stopping rule**, like the step limit in Lesson 4. Without it, a picky Reviewer and a stubborn Writer could loop forever.

The supervisor writes its decision to `next_agent`. A routing function reads it, just like `route_request` in Lesson 8. `finish` publishes the last draft and notes whether it was approved.

In [ ]:
def supervisor(state: ResearchState) -> dict:
    # Decide which agent should work next.
    review = state.get("review") or {}
    revision = state.get("revision", 0)

    if not state.get("notes"):
        next_agent = "researcher"
    elif not state.get("analysis"):
        next_agent = "analyst"
    elif not state.get("draft"):
        next_agent = "writer"
    elif review.get("draft_number") != revision:
        next_agent = "reviewer"          # the latest draft hasn't been reviewed yet
    elif review.get("approved"):
        next_agent = "finish"
    elif revision <= MAX_REVISIONS:
        next_agent = "writer"            # revise using the feedback
    else:
        next_agent = "finish"            # stop: revision limit reached

    print(f"[Supervisor] -> {next_agent}")
    return {"next_agent": next_agent}


def route_to_agent(state: ResearchState) -> str:
    # Routing function: follow the supervisor's decision.
    return state["next_agent"]


def finish(state: ResearchState) -> dict:
    # Publish the last draft as the final report.
    approved = state.get("review", {}).get("approved", False)
    note = "" if approved else "\n\n_Note: the revision limit was reached before the Reviewer approved this report._"
    return {"final_report": state["draft"] + note,
            "team_log": [f"Supervisor: finished ({'approved' if approved else 'revision limit reached'})."]}


print("Supervisor ready.")

Because the supervisor is plain code, we can **test every rule without calling the LLM at all**. Each test case is a hand-made state and the decision we expect.

In [ ]:
SUPERVISOR_CASES = [
    ("Nothing done yet", {}, "researcher"),
    ("Notes gathered", {"notes": "..."}, "analyst"),
    ("Analysis done", {"notes": "...", "analysis": {"outline": []}}, "writer"),
    ("First draft written", {"notes": "...", "analysis": {"x": 1}, "draft": "...", "revision": 1}, "reviewer"),
    ("Draft 1 rejected", {"notes": "...", "analysis": {"x": 1}, "draft": "...", "revision": 1,
                          "review": {"approved": False, "draft_number": 1}}, "writer"),
    ("Draft 2 not reviewed yet", {"notes": "...", "analysis": {"x": 1}, "draft": "...", "revision": 2,
                                  "review": {"approved": False, "draft_number": 1}}, "reviewer"),
    ("Draft 2 approved", {"notes": "...", "analysis": {"x": 1}, "draft": "...", "revision": 2,
                          "review": {"approved": True, "draft_number": 2}}, "finish"),
    ("Revision limit reached", {"notes": "...", "analysis": {"x": 1}, "draft": "...", "revision": MAX_REVISIONS + 1,
                                "review": {"approved": False, "draft_number": MAX_REVISIONS + 1}}, "finish"),
]


def check_supervisor() -> list[dict]:
    # Test the supervisor's rules with hand-made states. No API calls.
    print("Checking the supervisor's decisions:\n")
    results = []
    for name, partial_state, expected in SUPERVISOR_CASES:
        state = {**new_task("test"), **partial_state}
        got = supervisor(state)["next_agent"]
        ok = got == expected
        print(f"  {'PASS' if ok else 'FAIL'}  {name}: expected {expected}, got {got}\n")
        results.append({"case": name, "expected": expected, "got": got, "ok": ok})
    print(f"{sum(r['ok'] for r in results)}/{len(results)} supervisor checks passed.")
    return results


check_supervisor()

All 8 should pass. Testing the coordinator separately from the agents is one of the big advantages of a code-based supervisor.

## 12. Pass information between agents using shared state

Now we connect everything. The graph has the hub-and-spoke shape from Section 3:

- `START` → `supervisor`
- a **conditional edge** from `supervisor` to whichever agent it chose (or `finish`)
- a **normal edge** from every agent back to `supervisor`
- `finish` → `END`

In [ ]:
from langgraph.graph import StateGraph, START, END

AGENTS = ["researcher", "analyst", "writer", "reviewer"]


def build_graph():
    graph = StateGraph(ResearchState)

    graph.add_node("supervisor", supervisor)
    graph.add_node("researcher", researcher)
    graph.add_node("analyst", analyst)
    graph.add_node("writer", writer)
    graph.add_node("reviewer", reviewer)
    graph.add_node("finish", finish)

    graph.add_edge(START, "supervisor")
    graph.add_conditional_edges("supervisor", route_to_agent, AGENTS + ["finish"])
    for agent in AGENTS:
        graph.add_edge(agent, "supervisor")      # every agent reports back to the supervisor
    graph.add_edge("finish", END)

    return graph.compile()


app = build_graph()
print(app.get_graph().draw_mermaid())

Paste the diagram into [mermaid.live](https://mermaid.live). You should see the hub and spokes.

### Watch the information flow

`app.stream(..., stream_mode="updates")` runs the graph and shows **each step's update** as it happens: which node ran, and which fields of the shared state it wrote. This is the team communicating.

We set `recursion_limit` to 40 because each agent visit is two steps (the agent, then the supervisor), and a revision or two adds more.

In [ ]:
topic = "How do AI agents use memory?"

for step in app.stream(new_task(topic), {"recursion_limit": 40}, stream_mode="updates"):
    for node, update in step.items():
        written = [key for key in update if key != "team_log"]
        print(f"      {node:11} wrote -> {written}")

Read the output from top to bottom. The supervisor runs between every agent, and each agent writes only its own fields:

- `researcher` → `notes`
- `analyst` → `analysis`
- `writer` → `draft` and `revision`
- `reviewer` → `review`

If the Reviewer asked for changes, you'll see `writer` and `reviewer` run again. That's the revision loop.

## 13. Run and test the complete workflow

`run_team()` runs the whole team on one question, then prints the team log and the final report.

In [ ]:
def run_team(topic: str, app=None) -> dict:
    app = app or build_graph()
    print(f"\nResearch question: {topic}\n")
    result = app.invoke(new_task(topic), {"recursion_limit": 40})
    print("\n--- Team log ---")
    for line in result["team_log"]:
        print(" ", line)
    print("\n--- Final report ---\n")
    print(result["final_report"])
    return result


result = run_team("What is retrieval-augmented generation, and why does it reduce hallucinations?")

Now let's check the run automatically.

In [ ]:
log = " ".join(result["team_log"])

checks = [
    ("Every agent took part", all(name in log for name in ["Researcher", "Analyst", "Writer", "Reviewer"])),
    ("The Researcher used its tools", "using 0 searches" not in log),
    ("The final report has a Sources section", "source" in result["final_report"].lower()),
    ("Revisions stayed within the limit", result["revision"] <= MAX_REVISIONS + 1),
    ("The Reviewer approved the report", result["review"].get("approved", False)),
]

for name, ok in checks:
    print(("PASS " if ok else "CHECK"), name)
print(f"\n{sum(ok for _, ok in checks)}/{len(checks)} checks passed.")
print(f"\nDrafts written: {result['revision']}   Final review score: {result['review'].get('score')}/10")

### What to look for

- **The team log** reads like a story of the collaboration. Did the Reviewer ask for changes? Did the Writer fix them?
- **Sources:** does every fact in the report come from the Researcher's notes? Compare `result["notes"]` with the report.
- **Gaps:** look at `result["analysis"]["gaps"]`. Did the Analyst notice anything missing?

The last check ("Reviewer approved") can fail even when everything works correctly: the team stops at the revision limit and says so in the report. That's the stopping rule doing its job.

### Try it: your own research question

Change the question below and run the team. Questions that combine the course notes with general knowledge work best, for example "How do multi-agent systems handle safety?"

In [ ]:
my_result = run_team("How can AI agents be kept safe when they use tools?")

## 14. Challenge

### Part A: Add a new specialist

Add a **Summariser** agent that writes a three-bullet **TL;DR** (a very short summary) of the final report, for readers in a hurry.

1. Add a `summary` field to `ResearchState`.
2. Write a `summariser` node that reads `draft` and writes `summary`. A prompt template plus `llm` is enough.
3. Update the supervisor: once the Reviewer approves, send the work to the Summariser before `finish`. *Hint: add a rule before the "approved → finish" rule.*
4. Add the node to `build_graph`, add it to `AGENTS`, and update your `SUPERVISOR_CASES` so `check_supervisor()` still passes.
5. Run the team and print `result["summary"]`.

### Part B: An LLM supervisor

Replace the rule-based supervisor with an **LLM supervisor**: give the model a summary of the state (which fields are filled, the latest review) and ask it to choose the next agent with structured output, using a `Literal` of the agent names.

Keep **one rule in code**: if the revision limit is reached, always finish. Why is it a good idea to keep that rule in code instead of trusting the LLM?

### Stretch

Make the Researcher faster by splitting it into **two researchers that run in parallel**: one searches the course notes, the other searches Wikipedia. In LangGraph, two edges from the same node run their targets in parallel. Both should write to a list field that uses the `operator.add` reducer, so neither overwrites the other's notes.

In [ ]:
# Part A: your Summariser agent.

SUMMARY_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "TODO: tell the model to summarise the report in exactly three short bullet points."),
    ("human", "{draft}"),
])


def summariser(state):
    # TODO: run (SUMMARY_PROMPT | llm) on state["draft"], then return
    #       {"summary": reply.text, "team_log": ["Summariser: wrote the TL;DR."]}
    raise NotImplementedError("TODO: complete the summariser node")


# Then: add `summary: str` to ResearchState, update supervisor(), SUPERVISOR_CASES,
# AGENTS and build_graph(), re-run those cells, and test:
# app = build_graph()
# result = run_team("How do AI agents use memory?", app)
# print(result["summary"])

In [ ]:
# Part B: your LLM supervisor.

from typing import Literal


class SupervisorDecision(BaseModel):
    next_agent: Literal["researcher", "analyst", "writer", "reviewer", "finish"]
    reason: str = Field(description="One short sentence explaining the choice")


def llm_supervisor(state):
    # TODO 1: if state["revision"] > MAX_REVISIONS, return {"next_agent": "finish"} (the rule we keep in code)
    # TODO 2: describe the state for the LLM, e.g. "notes: done, analysis: missing, draft: missing, review: ..."
    # TODO 3: ask llm.with_structured_output(SupervisorDecision) to choose, print its reason,
    #         and return {"next_agent": decision.next_agent}
    raise NotImplementedError("TODO: complete llm_supervisor")


# Then use it in build_graph: graph.add_node("supervisor", llm_supervisor)

## Recap

| Concept | What you built |
|---|---|
| **Specialised agents** | Researcher (tool-using agent), Analyst and Reviewer (structured output), Writer (chain) |
| **Delegation** | The supervisor gives each task to the right specialist |
| **Supervisor architecture** | A hub-and-spoke graph: the supervisor in the middle, every agent reports back |
| **Agent communication** | Through shared state: each agent reads what it needs and writes only its own fields |
| **Shared state** | `ResearchState`, with an `operator.add` reducer for the team log |
| **Feedback loop** | Reviewer → Writer revisions, with a revision limit as the stopping rule |
| **Testing** | Supervisor rules tested without the LLM; the full team tested end to end |

**Next:** complete the starter file `lesson_09_multi_agent.py`, run its tests, and submit your work. See `README.md` for the submission checklist.